<a href="https://colab.research.google.com/github/adamarsyadfaizin/Pembelajaran-Mesin_2026_3F/blob/main/JS06/JS06_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas SVM

## 1. Klasifikasi voice.csv

Ketentuan:
- Split 70:30 dan 80:20
- Kernel linear, polynomial, dan RBF
- Performa dibandingkan berdasarkan akurasi

## 2. Klasifikasi Citra Siang dan Malam

Ketentuan:
- Dataset Praktikum 5
- Fitur histogram
- SVM kernel RBF
- Split 80:20
- Hyperparameter tuning RBF

## Bagian 1 — SVM pada voice.csv

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

### Load Dataset

In [10]:
voice_candidates = [
    Path('voice.csv'),
    Path('voice(1).csv'),
    Path('/content/voice.csv'),
    Path('/content/voice(1).csv')
]

voice_path = next((p for p in voice_candidates if p.exists()), None)
if voice_path is None:
    raise FileNotFoundError('Upload voice.csv terlebih dahulu.')

df = pd.read_csv(voice_path)
df.head()

,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [11]:
print('Shape:', df.shape)
print('Jumlah kelas:')
print(df['label'].value_counts())

Shape: (3168, 21)
Jumlah kelas:
label
male      1584
female    1584
Name: count, dtype: int64


### Pisahkan Fitur dan Label

In [12]:
X = df.drop(columns='label')
y = df['label']

print('Jumlah fitur:', X.shape[1])

Jumlah fitur: 20


### Training 6 Model

In [13]:
results = []

configs = [
    ('70:30', 0.30),
    ('80:20', 0.20)
]

kernels = ['linear', 'poly', 'rbf']

for split_name, test_size in configs:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    for kernel in kernels:
        model = make_pipeline(
            StandardScaler(),
            SVC(kernel=kernel)
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        results.append({
            'Split': split_name,
            'Kernel': kernel,
            'Accuracy': acc
        })

results_df = pd.DataFrame(results)
results_df

,Split,Kernel,Accuracy
0,70:30,linear,0.978970
1,70:30,poly,0.960042
2,70:30,rbf,0.983176
3,80:20,linear,0.974763
4,80:20,poly,0.955836
5,80:20,rbf,0.982650


### Tabulasi Akurasi

In [14]:
accuracy_table = results_df.pivot(index='Kernel', columns='Split', values='Accuracy')
accuracy_table = (accuracy_table * 100).round(2)
accuracy_table.columns = [f'Akurasi {c} (%)' for c in accuracy_table.columns]
accuracy_table

,Akurasi 70:30 (%),Akurasi 80:20 (%)
Kernel,,
linear,97.90,97.48
poly,96.00,95.58
rbf,98.32,98.26


In [15]:
best_voice = results_df.loc[results_df['Accuracy'].idxmax()]
print('Model terbaik:')
print('Split  :', best_voice['Split'])
print('Kernel :', best_voice['Kernel'])
print(f"Akurasi: {best_voice['Accuracy']*100:.2f}%")

Model terbaik:
Split  : 70:30
Kernel : rbf
Akurasi: 98.32%


---
## Bagian 2 — Klasifikasi Siang dan Malam dengan Fitur Histogram

In [16]:
import os
import zipfile
import cv2

# Ekstrak images.zip jika folder images belum ada
if not Path('images').exists():
    image_zip_candidates = [Path('images.zip'), Path('/content/images.zip')]
    image_zip = next((p for p in image_zip_candidates if p.exists()), None)

    if image_zip is None:
        raise FileNotFoundError('Upload images.zip dari Praktikum 5 terlebih dahulu.')

    with zipfile.ZipFile(image_zip, 'r') as zip_ref:
        zip_ref.extractall('.')

print('Folder images ditemukan:', Path('images').exists())

Folder images ditemukan: True


### Load Seluruh Citra

In [17]:
def collect_images(root='images'):
    image_paths = []
    labels = []

    for split in ['training', 'test']:
        for label in ['day', 'night']:
            folder = Path(root) / split / label
            for file in sorted(folder.glob('*.jpg')):
                image_paths.append(str(file))
                labels.append(label)

    return image_paths, np.array(labels)

image_paths, image_labels = collect_images()

print('Jumlah citra:', len(image_paths))
print(pd.Series(image_labels).value_counts())

Jumlah citra: 400
day      200
night    200
Name: count, dtype: int64


### Ekstraksi Fitur Histogram

In [18]:
def extract_histogram(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f'Gagal membaca: {image_path}')

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
    hist = hist / (hist.sum() + 1e-12)
    return hist

X_hist = np.array([extract_histogram(path) for path in image_paths])
y_hist = image_labels

print('Shape fitur histogram:', X_hist.shape)

Shape fitur histogram: (400, 256)


### Split 80:20

In [19]:
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_hist, y_hist,
    test_size=0.20,
    random_state=42,
    stratify=y_hist
)

print('Data train:', X_train_img.shape[0])
print('Data test :', X_test_img.shape[0])

Data train: 320
Data test : 80


### SVM RBF + Hyperparameter Tuning

In [20]:
rbf_pipeline = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf')
)

param_grid = {
    'svc__C': [0.1, 1, 10, 50, 100],
    'svc__gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

grid = GridSearchCV(
    rbf_pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_img, y_train_img)

print('Best parameter:', grid.best_params_)
print(f'Best CV accuracy: {grid.best_score_*100:.2f}%')

Best parameter: {'svc__C': 10, 'svc__gamma': 'scale'}
Best CV accuracy: 98.44%


### Evaluasi Data Test

In [21]:
best_rbf = grid.best_estimator_
y_pred_img = best_rbf.predict(X_test_img)
acc_img = accuracy_score(y_test_img, y_pred_img)

print(f'Akurasi test SVM RBF + histogram: {acc_img*100:.2f}%')

Akurasi test SVM RBF + histogram: 100.00%


## Kesimpulan

In [22]:
print('=== HASIL VOICE.CSV ===')
print(accuracy_table)
print()
print('=== HASIL CITRA SIANG/MALAM ===')
print('Best parameter:', grid.best_params_)
print(f'Akurasi test: {acc_img*100:.2f}%')

=== HASIL VOICE.CSV ===
        Akurasi 70:30 (%)  Akurasi 80:20 (%)
Kernel                                      
linear              97.90              97.48
poly                96.00              95.58
rbf                 98.32              98.26

=== HASIL CITRA SIANG/MALAM ===
Best parameter: {'svc__C': 10, 'svc__gamma': 'scale'}
Akurasi test: 100.00%
